# TP Final CD.05 - Herramientas para Grandes Volúmenes de Datos
## Bruno Bustos

Clasificación multiclase de etapas de cultivo a partir de sensores IoT agrícolas.

**Índice:**
1. Descripción del dataset y justificación
2. Preparación de datos con PySpark
3. Experimentación con MLflow + Optuna
4. Monitoreo con Evidently
5. Interpretabilidad con SHAP
6. Selección del modelo final y registro en Model Registry
7. Invocación del serving endpoint
8. Conclusiones adicionales
9. Instrucciones para reproducir

## Setup: instalación de dependencias

In [0]:
%pip install --upgrade pydantic
%pip uninstall -y evidently
%pip install evidently optuna shap requests

In [0]:
%restart_python

## Configuración general (catálogo, schema, rutas)


In [0]:
CATALOG = "workspace"
SCHEMA = "tp_final_bigdata"
RAW_DATA_PATH = "/Workspace/Repos/brbustos@itba.edu.ar/TP_BigData_BrunoBustos/IOT_Agricultura_Dataset.csv" 
EXPERIMENT_NAME = "/Users/brbustos@itba.edu.ar/TP_Final_Exp"

TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.iot_agri_train"
TEST_TABLE = f"{CATALOG}.{SCHEMA}.iot_agri_test"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.iot_agriculture_rf_classifier"
ENDPOINT_NAME = "iot-agriculture-rf-endpoint"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# 1. Descripción del Dataset y Justificación

### Fuente y descripción

- **Nombre:** Advanced IoT Agriculture 2024

- **Fuente:** [Kaggle - wisam1985/advanced-iot-agriculture-2024](https://www.kaggle.com/datasets/wisam1985/advanced-iot-agriculture-2024)

- **Origen:** mediciones de sensores IoT sobre cultivos, en el marco de experimentación agrícola (Tikrit University).

- **Tamaño:** 30.000 filas x 14 columnas.

- **Variable objetivo:** `class`, con 6 clases perfectamente balanceadas (5.000 filas c/u): `SA`, `SB`, `SC`, `TA`, `TB`, `TC` (combinación de tipo de tratamiento/planta y etapa de crecimiento). Es un problema de **clasificación multiclase**.

- **Features:** 12 mediciones numéricas de sensores/biometría de la planta (clorofila, altura, peso húmedo/seco, área foliar, número de hojas, diámetro y longitud de raíz porcentaje de materia seca vegetativa y de raíz, entre otras) más `random` (bloque/réplica experimental categórico: R1/R2/R3).

- **Calidad:** sin valores nulos

### Adecuación al curso

- **Volumen:** 30.000 filas. Tal vez no es lo más "big data" en sentido del tamaño del dataset para flexibilisar las iteraciones, pero los datos provienen de sensores IoT que en una situación real, generan lecturas de forma continua (o data streaming), escalando a millones de registros. Esto justifica usar PySpark/Databricks por la arquitectura pensada para ese volumen de producción.

- **Multiclase balanceado**: permite usar métricas estándar (ROC AUC macro, F1 macro) sin la complejidad adicional de manejar desbalance de clases.

- **Features numéricas de sensores**: dataset ideal para Random Forest (no requiere escalado) y para SHAP (interpretar qué mediciones de sensores más influyen).

- **Naturaleza IoT**: encaja naturalmente con el caso de uso de Evidently (monitoreo de drift), ya que en producción real sensores físicos van a derivar con el tiempo.

### Limitaciones

- Las clases perfectamente balanceadas sugieren un dataset **sintético o simulado** — el
  drift observado en Evidently probablemente sea menor al que se vería con sensores reales.
- El volumen (30k filas) es modesto frente a big data real; el valor pedagógico está en la
  arquitectura del pipeline, reproducible sin cambios ante un volumen mucho mayor.

### Exploración visual rápida (pandas)

Usamos pandas solo para esta exploración inicial (30k filas lo permite cómodamente); el
procesamiento productivo de la Sección 2 en adelante se hace con PySpark.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

df_pandas_preview = pd.read_csv(RAW_DATA_PATH)
df_pandas_preview.columns = [
    c.strip().lower().replace(" ", "_").replace("(", "").replace(")", "")
    for c in df_pandas_preview.columns
]

print(f"Filas: {len(df_pandas_preview):,} | Columnas: {len(df_pandas_preview.columns)}")
df_pandas_preview.head()

In [0]:
df_pandas_preview["class"].value_counts().sort_index().plot(
    kind="bar", title="Distribución de clases", figsize=(6, 4)
)
plt.ylabel("Cantidad de filas")
plt.show()

In [0]:
numeric_cols_preview = df_pandas_preview.select_dtypes(include="number").columns
df_pandas_preview[numeric_cols_preview].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(10, 8))
corr = df_pandas_preview[numeric_cols_preview].corr()
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(numeric_cols_preview)), numeric_cols_preview, rotation=90)
plt.yticks(range(len(numeric_cols_preview)), numeric_cols_preview)
plt.colorbar()
plt.title("Correlación entre features numéricas")
plt.show()

# 2. Preparación de Datos con PySpark

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def normalize_column_names(df: DataFrame) -> DataFrame:
    renamed = df
    for col_name in df.columns:
        new_name = (
            col_name.strip()
            .lower()
            .replace("  ", " ")
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
        )
        renamed = renamed.withColumnRenamed(col_name, new_name)
    return renamed


df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_DATA_PATH)
)

df = normalize_column_names(df_raw)
print(f"Filas: {df.count():,} | Columnas: {len(df.columns)}")
df.printSchema()

### Control de calidad de datos

In [0]:
# Nulls por columna
null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
display(null_counts)

In [0]:
# Duplicados exactos
total_rows = df.count()
distinct_rows = df.distinct().count()
print(f"Filas totales: {total_rows:,} | Únicas: {distinct_rows:,} | Duplicados: {total_rows - distinct_rows:,}")

In [0]:
# Balance de clases y de la columna 'random'
display(df.groupBy("class").count().orderBy("class"))
display(df.groupBy("random").count().orderBy("random"))

In [0]:
# Estadísticas descriptivas
numeric_cols = [c for c, t in df.dtypes if t in ("double", "float", "int", "bigint")]
display(df.select(numeric_cols).describe())

In [0]:
# Outliers Rango Interquartil (IQR)
def count_outliers_iqr(df: DataFrame, numeric_cols: list) -> dict:
    """Cuenta outliers por columna usando el criterio de rango intercuartílico (1.5*IQR)."""
    outlier_counts = {}
    quantiles = df.approxQuantile(numeric_cols, [0.25, 0.75], 0.01)
    for col_name, (q1, q3) in zip(numeric_cols, quantiles):
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = df.filter((F.col(col_name) < lower) | (F.col(col_name) > upper)).count()
        outlier_counts[col_name] = count
    return outlier_counts


outlier_summary = count_outliers_iqr(df, numeric_cols)
for col_name, count in outlier_summary.items():
    print(f"{col_name}: {count} outliers")

### Encoding de variables categóricas y split train/test

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

label_indexer = StringIndexer(inputCol="class", outputCol="label", stringOrderType="alphabetAsc")
random_indexer = StringIndexer(inputCol="random", outputCol="random_idx")
random_encoder = OneHotEncoder(inputCol="random_idx", outputCol="random_ohe")

encoding_pipeline = Pipeline(stages=[label_indexer, random_indexer, random_encoder])
df_encoded = encoding_pipeline.fit(df).transform(df)


train_df, test_df = df_encoded.randomSplit([0.8, 0.2], seed=44102564) # Seed - DNI
print(f"Train: {train_df.count():,} filas | Test: {test_df.count():,} filas")

display(train_df.groupBy("class").count().orderBy("class"))
display(test_df.groupBy("class").count().orderBy("class"))

### Persistencia en Delta (Unity Catalog)

In [0]:
train_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TRAIN_TABLE)
test_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TEST_TABLE)
print(f"Guardado: {TRAIN_TABLE}\nGuardado: {TEST_TABLE}")

# 3. Experimentación con MLflow + Optuna

Convertimos train/test a pandas (volumen manejable, ~30k filas) para trabajar con
scikit-learn. Definimos acá `FEATURE_COLS` y `CLASS_NAMES`, reutilizados en el resto de
la notebook (Evidently, SHAP, selección final e invocación del endpoint).

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

train_pdf = train_df.toPandas()
test_pdf = test_df.toPandas()

FEATURE_COLS = [
    c for c in train_pdf.columns
    if c not in ("class", "label", "random", "random_idx", "random_ohe")
] + ["random_idx"]

CLASS_NAMES = sorted(train_pdf["class"].unique())  

X_train, y_train = train_pdf[FEATURE_COLS], train_pdf["label"].astype(int)
X_test, y_test = test_pdf[FEATURE_COLS], test_pdf["label"].astype(int)

print(f"Train: {X_train.shape} | Test: {X_test.shape} | Clases: {CLASS_NAMES}")

### Grid search con nested runs (≥4 corridas)

Métricas adaptadas a multiclase: `roc_auc_score` con `multi_class="ovr"` sobre la matriz
completa de probabilidades, y precisión/recall/F1 tomados de `"macro avg"`.

In [0]:
mlflow.set_experiment(EXPERIMENT_NAME)

param_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9},
]

best_auc, best_run_id, best_clf = 0.0, None, None

with mlflow.start_run(run_name="parent_run") as parent_run:
    mlflow.log_param("experiment_type", "RandomForest_Hyperparameter_Search")
    for params in param_grid:
        with mlflow.start_run(run_name=f"child_{params['n_estimators']}_{params['max_depth']}", nested=True) as run:
            clf = RandomForestClassifier(random_state=42, **params)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            y_proba = clf.predict_proba(X_test)

            acc = accuracy_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
            cm = confusion_matrix(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)
            signature = infer_signature(X_test, y_pred)

            mlflow.log_params(params)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("roc_auc_macro", auc)
            mlflow.log_metric("precision_macro", report["macro avg"]["precision"])
            mlflow.log_metric("recall_macro", report["macro avg"]["recall"])
            mlflow.log_metric("f1_macro", report["macro avg"]["f1-score"])
            mlflow.sklearn.log_model(clf, name="model", signature=signature)
            mlflow.log_dict(report, "classification_report.json")
            mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")

            if auc > best_auc:
                best_auc, best_run_id, best_clf = auc, run.info.run_id, clf

print(f"Mejor corrida (grid): {best_run_id} | ROC AUC macro: {best_auc:.4f}")

### Optimización con Optuna

In [0]:
import optuna

mlflow.set_experiment(f"{EXPERIMENT_NAME}_Optuna")


def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200, step=50)
    max_depth = trial.suggest_int("max_depth", 3, 9, step=2)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

    clf = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
        random_state=42,
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    signature = infer_signature(X_test, y_pred)

    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}"):
        mlflow.log_params({
            "n_estimators": n_estimators, "max_depth": max_depth,
            "min_samples_split": min_samples_split, "min_samples_leaf": min_samples_leaf,
        })
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc_macro", auc)
        mlflow.log_metric("precision_macro", report["macro avg"]["precision"])
        mlflow.log_metric("recall_macro", report["macro avg"]["recall"])
        mlflow.log_metric("f1_macro", report["macro avg"]["f1-score"])
        mlflow.sklearn.log_model(clf, name="model", signature=signature)
        mlflow.log_dict(report, "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")

    trial.set_user_attr("clf", clf)
    return auc


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

best_optuna_trial = max(study.trials, key=lambda t: t.value)
if best_optuna_trial.value > best_auc:
    best_auc = best_optuna_trial.value
    best_clf = best_optuna_trial.user_attrs["clf"]

print(f"Mejor ROC AUC macro (grid + Optuna): {best_auc:.4f}")

In [0]:
import optuna.visualization
from optuna.importance import MeanDecreaseImpurityImportanceEvaluator

optuna.visualization.plot_optimization_history(study)

try:
    optuna.visualization.plot_param_importances(
        study, evaluator=MeanDecreaseImpurityImportanceEvaluator()
    )
except RuntimeError as e:
    print(f"No se pudo calcular la importancia de hiperparámetros: {e}")
    print("Esto suele pasar cuando los trials tienen muy poca variación en el objetivo (ROC AUC).")

optuna.visualization.plot_parallel_coordinate(study)
optuna.visualization.plot_slice(study)

# 4. Monitoreo con Evidently

Comparamos train (referencia) vs. test (actual) usando el **mejor modelo** (`best_clf`),
con `MulticlassClassification` (6 clases). Solo pasamos `prediction_labels` (sin
probabilidades por clase) para simplificar.

In [0]:
import numpy as np
np.float_ = np.float64

from evidently import Report
from evidently.presets import DataDriftPreset, ClassificationPreset
from evidently.future.datasets import Dataset, DataDefinition
from evidently import MulticlassClassification

y_test_pred = best_clf.predict(X_test)
y_train_pred = best_clf.predict(X_train)

data_current = X_test.copy()
data_current["target"] = y_test.values
data_current["prediction"] = y_test_pred

data_reference = X_train.copy()
data_reference["target"] = y_train.values
data_reference["prediction"] = y_train_pred

data_definition = DataDefinition(
    classification=[MulticlassClassification(target="target", prediction_labels="prediction")]
)

current_dataset = Dataset.from_pandas(data_current, data_definition=data_definition)
reference_dataset = Dataset.from_pandas(data_reference, data_definition=data_definition)

report = Report(metrics=[DataDriftPreset(), ClassificationPreset()])
snapshot = report.run(current_data=current_dataset, reference_data=reference_dataset)
snapshot.save_html("resultados_evidently.html")

#  Conclusiones del reporte de Evidently

---

## 1. Dataset Drift: inexistente
- **0 de 15 columnas** con drift  
- Share drifted: **0.0 (0%)**  
- El reporte indica: *Dataset Drift is NOT detected*

![](/Workspace/Repos/brbustos@itba.edu.ar/TP_BigData_BrunoBustos/Imagenes_Evidently/Evidently_1.png)

**Conclusión:**  
✔️ Los datos en producción son **identicos a los de entrenamiento**  
✔️ No hay problema de distribución

---


## 2. Performance del modelo: sigue siendo muy alta

### Modelo actual:
- Accuracy: **1.0**
- Precision: **1.0**
- Recall: **1.0**
- F1: **1.0**
- ROC AUC: **1.0**

![](/Workspace/Repos/brbustos@itba.edu.ar/TP_BigData_BrunoBustos/Imagenes_Evidently/Evidently_2.png)

**Conclusión:**  
✔️ El modelo funciona de forma excelente
pero invita a cuestionar si es una desventaja del dataset elegido al ser potencialmente artificial por la inexistencia de ruido en los valores.

---

## 3. Degradación vs modelo de referencia

### Modelo de referencia:
- Métricas: **1.0 (perfecto)**

### Modelo actual:
- No existe caída en las métricas

**Conclusión:**  
✔️ No hay drift ni ruido en los datos reales. 

---

## 4. Confusion Matrix: errores controlados
- Ningun falso positivo ni negativo encontrado en los resultados  
- Clasificación ideal  

![](/Workspace/Repos/brbustos@itba.edu.ar/TP_BigData_BrunoBustos/Imagenes_Evidently/Evidently_3.png)

**Conclusión:**  
✔️ No hay problema operativo alguno  

---

## 5. Precision-Recall Curve estable
- Curva óptima 
- No hay colapso del modelo  

![](/Workspace/Repos/brbustos@itba.edu.ar/TP_BigData_BrunoBustos/Imagenes_Evidently/Evidently_4.png)

**Conclusión:**  
✔️ Precisión y recall ideales

---


# 5. Interpretabilidad con SHAP

Con 6 clases, `shap_values` tiene una dimensión extra (muestras × features × clases) —
iteramos sobre las clases en vez de fijar una sola como en el caso binario.

In [0]:
import shap

X_train_sample = X_train.sample(n=min(500, len(X_train)), random_state=42)
explainer = shap.Explainer(best_clf, X_train_sample)
shap_values = explainer(X_test)

In [0]:
for i, class_name in enumerate(CLASS_NAMES):
    print(f"--- Clase: {class_name} ---")
    shap.summary_plot(shap_values.values[:, :, i], X_test, feature_names=X_test.columns, show=True)

In [0]:
sample_idx = 0
predicted_class_idx = int(best_clf.predict(X_test.iloc[[sample_idx]])[0])
shap.plots.waterfall(shap_values[sample_idx, :, predicted_class_idx])

In [0]:
import matplotlib.pyplot as plt

importances = best_clf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
plt.figure(figsize=(8, 6))
plt.barh(np.array(X_test.columns)[sorted_idx], importances[sorted_idx])
plt.xlabel("Feature Importance")
plt.title("Feature Importance en clasificador de Random Forest")
plt.gca().invert_yaxis()
plt.show()

# 6. Selección del Modelo Final y Registro en Model Registry

In [0]:
runs_grid = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
runs_optuna = mlflow.search_runs(experiment_names=[f"{EXPERIMENT_NAME}_Optuna"])
all_runs = pd.concat([runs_grid, runs_optuna], ignore_index=True)

comparison_cols = [
    "run_id", "tags.mlflow.runName",
    "metrics.roc_auc_macro", "metrics.accuracy",
    "metrics.precision_macro", "metrics.recall_macro", "metrics.f1_macro",
    "params.n_estimators", "params.max_depth",
]
comparison_cols = [c for c in comparison_cols if c in all_runs.columns]

leaderboard = (
    all_runs[all_runs["metrics.roc_auc_macro"].notna()][comparison_cols]
    .sort_values("metrics.roc_auc_macro", ascending=False)
    .reset_index(drop=True)
)
display(leaderboard)

In [0]:
best_row = leaderboard.iloc[0]
final_run_id = best_row["run_id"]

print(f"Run ganador: {final_run_id} | ROC AUC macro: {best_row['metrics.roc_auc_macro']:.4f}")
print(f"Accuracy: {best_row['metrics.accuracy']:.4f} | F1 macro: {best_row['metrics.f1_macro']:.4f}")

### Justificación de la elección 

- Run Ganador: 909ac8c2a42e4a5797fc82bea00c08f8
- Tag MLFlow: child_200_9
- _ROC AUC macro_: 1.0000
- _Accuracy_: 1.0000
- _F1 macro_: 1.0000

- El modelo ganador alcanzó **ROC AUC macro = 1.0** en todas las configuraciones evaluadas (grid search y Optuna), con matriz de confusión perfectamente diagonal sobre el conjunto de test. 

- La separación perfecta se explica por la combinación conjunta de las 12 mediciones biométricas, consistente con un dataset generado sintéticamente.

### Registro en Unity Catalog Model Registry

In [0]:
mlflow.set_registry_uri("databricks-uc")

model_uri = f"runs:/{final_run_id}/model"
registered_model = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
print(f"Modelo registrado: {MODEL_NAME} - versión {registered_model.version}")

In [0]:
from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(name=MODEL_NAME, alias="champion", version=registered_model.version)
print(f"Alias 'champion' asignado a {MODEL_NAME} versión {registered_model.version}")

### Despliegue como Serving Endpoint

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()

served_entity = ServedEntityInput(
    entity_name=MODEL_NAME,
    entity_version=str(registered_model.version),
    workload_size="Small",
    scale_to_zero_enabled=True,
)

existing_endpoints = [e.name for e in w.serving_endpoints.list()]

if ENDPOINT_NAME in existing_endpoints:
    w.serving_endpoints.update_config_and_wait(name=ENDPOINT_NAME, served_entities=[served_entity])
    print(f"Endpoint '{ENDPOINT_NAME}' actualizado a la versión {registered_model.version}")
else:
    w.serving_endpoints.create_and_wait(name=ENDPOINT_NAME, config=EndpointCoreConfigInput(served_entities=[served_entity]))
    print(f"Endpoint '{ENDPOINT_NAME}' creado con la versión {registered_model.version}")

# 7. Invocación del Serving Endpoint

Tomamos host y token del contexto del notebook (no hace falta archivo de token manual).
Reutilizamos `X_test`/`y_test` y `CLASS_NAMES` ya definidos en la Sección 3.

In [0]:
import json
import requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()
SERVING_URL = f"{DATABRICKS_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"


def create_tf_serving_json(data):
    return {"inputs": {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}


def score_model(dataset):
    headers = {"Authorization": f"Bearer {DATABRICKS_TOKEN}", "Content-Type": "application/json"}
    ds_dict = (
        {"dataframe_split": dataset.to_dict(orient="split")}
        if isinstance(dataset, pd.DataFrame)
        else create_tf_serving_json(dataset)
    )
    response = requests.post(url=SERVING_URL, headers=headers, data=json.dumps(ds_dict, allow_nan=True))
    if response.status_code != 200:
        raise Exception(f"Request failed with status {response.status_code}, {response.text}")
    return response.json()

In [0]:
# Ejemplo #1: una sola fila
fila_ejemplo = X_test.iloc[[0]]
clase_real_ejemplo = test_pdf["class"].iloc[0]

resultado = score_model(fila_ejemplo)
print("Respuesta del endpoint:", resultado)
print(f"Clase real: {clase_real_ejemplo}")

In [0]:
# Ejemplo #2: batch de 10 filas, comparando contra la clase real
batch = X_test.iloc[:10]
clases_reales = test_pdf["class"].iloc[:10].tolist()

resultado_batch = score_model(batch)
predicciones = resultado_batch.get("predictions", resultado_batch)

comparacion = pd.DataFrame({
    "clase_real": clases_reales,
    "prediccion_endpoint": [CLASS_NAMES[int(p)] for p in predicciones],
})
comparacion["correcto"] = comparacion["clase_real"] == comparacion["prediccion_endpoint"]
display(comparacion)

# 8. Conclusiones Adicionales


- Resumen de calidad de datos (Sección 2): duplicados, outliers detectados.
    - Realizando la deteccion de outliers usando el rango intercuartil, se detectaron outliers en 2 de las 12 features: **average_wet_weight_of_the_root_awwr** (1.627 filas) y **percentage_of_dry_matter_for_root_growth_pdmrg** (1.646 filas). El análisis evidenció que estos outliers no son errores de medición sino variaciones entre clases: por ejemplo, la media de **awwr** va de 1.45 (clase TB) a 4,86 (clase SC), con desvíos estándar muy dispares (0,08 a 1,51) — un valor normal para una clase resulta estadísticamente extremo para otra al medirlo contra la distribución global. EL Random Forest no se ve afectado por esta característica del dataset.
- Tabla comparativa de corridas y modelo final elegido, con justificación (Sección 6).
    - Tag MLFlow: **child_200_9** (1er Run)
    - Justificación: Debido a que los resultados de todas las iteraciones fueron perfectos (1), se puede tomar el primer resultado como el adecuado.

- Variables más importantes según SHAP (Sección 5).
    - Feature mas importante en Random Forest : **plant_height_rate_phr**
- Resultado de la invocación del endpoint (Sección 7).
    - Clase Real: **SA**

- Limitaciones del dataset (Sección 1) y posibles mejoras futuras.
    - El ROC AUC de 1.0 en todas las corridas no es representativo de un escenario de real porque los sensores en campo tendróan algún tipo de ruido y/o solapamiento entre clases que este dataset no presenta. El valor del trabajo presente está en la arquitectura del pipeline (PySpark, MLflow, Model Registry, Evidently, SHAP), que es igual de válida ante un dataset con separación menos ideal o cercana a 1.



## 9. Instrucciones para Reproducir

-- En Databricks
1. Subir **Advanced_IoT_Dataset.csv** (encontrado en el repo o en enlace de Kaggle) a un Volume de Unity Catalog (o DBFS).
2. Importar la notebook **TP_Final_BigData_BrunoBustos.ipynb** al workspace (Workspace → Import → File).
3. En la celda de configuración general (arriba de todo), ajustar _RAW_DATA_PATH_ (ruta del CSV subido) y _EXPERIMENT_NAME_ (tu path de usuario de Databricks).
4. Correr la notebook completa